# v001_base_model — base model: normalise, multi-pass blocking, 47 features, LightGBM, tuned 1-to-1 rule

| Field | Value |
|---|---|
| **Version** | `v001_base_model` |
| **Plan group** | INT (integrated V1: B1–B4, A1–A5, C1–C5, D3, E1–E2) |
| **Parent version** | — (first version) |
| **Author** | M1 rajaguru2004 |
| **Date** | 2026-09-25 |
| **Status** | shortlisted |

The first complete pipeline of the research plan (`docs/plan/00_MASTER_PLAN.md` §4, the V1
tier), built for the first leaderboard upload. Every stage is library code in
`src/entity_resolution/` (tested, documented); this notebook only configures, runs and
inspects it:

```
raw TSV -> normalize (static rules + learned transliteration map) -> blocking (exact keys,
name+address word TF-IDF, short-address name char-grams, per country) -> features (47 pair
similarities, chunked) -> LightGBM matcher -> decision rule tuned for macro F0.5 on the tune
split (pool-side 1-to-1) -> matching_results.tsv + candidate_pairs.tsv
```

House rules: every code cell is preceded by a markdown cell saying what it does and why;
the only decision metric is macro F0.5 on the fixed validation fold.

## 1. Hypothesis

* **Change vs parent:** none, first version. Replaces the plan's heuristic walking skeleton
  with the full V1 stack in one step, because the target for the first upload is a
  competitive score, not a calibration point.
* **Why it should score high:** measured on the data (01 §3), names are ambiguous (48 % of
  S1 core names are shared) but addresses separate decoys; true pairs always share the
  country; 15 % of true pairs share no name token, mostly Indic-script names that a learned
  token map turns back into English words. So: recall from name **and** address channels
  inside the country partition, precision from address-aware features, a boosted model and a
  conservative 1-to-1 set rule.
* **Expected effect:** candidate recall ≥ 0.98 at ≤ 40 candidates per S1; local macro
  F0.5 ≥ 0.97; singleton F0.5 ≥ 0.9.
* **Discard if:** local macro F0.5 < 0.95 (then the next version isolates the weak stage
  from the error analysis in §6).

## 2. Setup

Imports from the shared library, this experiment's folders and the pipeline configuration.
`PipelineConfig()` holds every choice of this version (its defaults *are* the V1 plan);
it is saved to `artifacts/config.json` and logged in `metrics.json`. `timings` collects the
stage run times under the eight standard labels (13 §2.2).

In [1]:
import json
import subprocess
import sys
import time
from dataclasses import asdict

import numpy as np
import pandas as pd

from entity_resolution import config as C
from entity_resolution.blocking import PASS_BITS
from entity_resolution.evaluate import error_samples, harder_fold, pair_in, slice_report
from entity_resolution.features import FEATURE_COLUMNS, feature_names
from entity_resolution.normalize import NORM_COLUMNS, normalise_records
from entity_resolution.pipeline import (
    PipelineConfig, fit, load_normalised, peak_rss_gb, run_fold, run_test,
)
from entity_resolution.split import load_fold
from entity_resolution.tracking import log_result, timed

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.max_columns", 30)

EXP_DIR = C.EXPERIMENTS / "v001_base_model"
ARTIFACTS = EXP_DIR / "artifacts"
cfg = PipelineConfig()
timings: dict[str, float] = {}
t_start = time.time()
print(json.dumps(cfg.record(), indent=1)[:3000])

{
 "normalise": {
  "transliterate": true,
  "strip_legal": true,
  "expand_abbrev": true,
  "region_map": null,
  "chunk_rows": 1000000,
  "learn_token_map": true,
  "token_map_min_count": 3,
  "token_map_min_share": 0.5
 },
 "blocking": {
  "exact_keys": [
   "name_core",
   "name_sorted",
   "name_squash"
  ],
  "exact_max_group": 50,
  "name_char": {
   "column": "name_core",
   "analyzer": "char_wb",
   "ngram": [
    3,
    3
   ],
   "top_k": 10,
   "min_sim": 0.5,
   "max_df": 0.2,
   "min_df": 2,
   "sublinear_tf": true,
   "pool_max_addr_tokens": 3,
   "max_df_abs": 20000
  },
  "name_addr_word": {
   "column": "name_addr",
   "analyzer": "word",
   "ngram": [
    1,
    2
   ],
   "top_k": 25,
   "min_sim": 0.2,
   "max_df": 0.01,
   "min_df": 2,
   "sublinear_tf": true,
   "pool_max_addr_tokens": null,
   "max_df_abs": 10000
  },
  "addr_char": null,
  "max_per_s1": 60,
  "s1_chunk": 50000,
  "n_threads": 12,
  "vocab_sample": 500000,
  "seed": 42
 },
 "feature_groups": [
 

## 3. Data

The fixed validation split (`split.load_fold`, 20 % of Source 1 by id hash, seed 42): every
version scores the same held-out entities against the same pool, so local F0.5 values are
comparable. Everything trainable is fitted on the `train` fold only; inside it,
`trainset.inner_split` (seed 4242, 25 % tune) separates model training (fit side) from
early stopping and threshold tuning (tune side). The val pool keeps the records matched to
*other* val entities: they are the decoys that make singletons and same-name businesses hard.

In [2]:
with timed("load", timings):
    train = load_fold("train", columns=[])   # ids only: the pipeline reads normalised records
    val = load_fold("val")                   # raw columns kept for the error samples
pd.DataFrame([train.summary(), val.summary()])

,fold,s1,s2,s3,true_pairs,singleton_share
0,train,1765488,4026279,4229875,6110753,0.0558
1,val,441333,1008337,1055728,1527612,0.0559


## 4. Method

### 4.1 Normalisation (`normalize.py`, `token_maps.py`)

Rules, in order (05 §2), each undoing a noise pattern measured on true pairs:

* **Transliteration** (`anyascii`, ISC): rows with non-ASCII characters are transliterated on
  the raw text (Indic scripts, accents). 7 % of pool names and 9 % of pool addresses are in an
  Indic script; Source 1 is all Latin.
* **Case, `&` → `and`, punctuation, apostrophes, dotted initials** (`L.L.C.` → `llc`).
* **Domain / handle forms** (`allh0spitalityproducts.com`, `@sakashpoint`,
  `ORTHOPEDICHEALTHCOM`) lose their suffix; **leet digits** inside words fold to letters
  (`F0rman` → `forman`).
* **Legal forms** leave `name_core` and are kept, canonical, in `legal_form` (`Pvt. Ltd.` =
  `Private Limited` = `pvt ltd`; SARL/SAS/EURL for France). **Honorific prefixes** injected
  into pool names (`Mr`, `Smt`, `Shri`) are dropped.
* **Learned transliteration map**: for true pairs of the train fold whose pool name is in an
  Indic script, tokens are aligned by position with the Latin name (`सॉल्यूशंस` →
  `solyusms` ↔ `solutions`); tokens seen ≥ 3 times with a ≥ 50 % consistent partner form a
  map applied to non-Latin names. Learned from the provided training pairs only.
* **Addresses**: region components (full name, code or native-script name: `Maharashtra`,
  `MH`, `महाराष्ट्र`) become one code; ordinals lose their suffix; digits split from letters;
  leading zeros go; street types are canonicalised (`Street`/`St`/`Saint` — the generator
  writes `Saint` for `St`); old/new Indian city names are unified.

Derived keys: `name_first`, `name_sorted` (sorted distinct tokens), `name_squash` (letters and
digits only), `addr_nums`, `postcode`, `region`, `name_addr`. The cell shows raw and
normalised val records, including Indic-script names (static rules only: the learned map is
fitted inside `pipeline.fit` in §4.4, which prints examples of it).

In [3]:
examples = pd.concat([val.s2[val.s2[C.NAME].str.contains(r"[\x{0900}-\x{0DFF}]", regex=True)].head(3),
                      val.s3.sample(5, random_state=1)], ignore_index=True)
normalise_records(examples)[["entity_id", "name_norm", "name_core", "legal_form", "name_squash",
                             "addr_norm", "region"]].assign(raw_name=examples[C.NAME].to_numpy())

,entity_id,name_norm,name_core,legal_form,name_squash,addr_norm,region,raw_name
0,S2-322802571,hotl vemcrs limited,hotl vemcrs,ltd,hotlvemcrs,lucknow c 121 meena bakery chauraha nr dariyapur up,up,होटल वेंचर्स लिमिटेड
1,S2-669353485,lksmi devlprs praivet limited,lksmi devlprs,pvt ltd,lksmidevlprs,h no 1338 nesari tal gadhinglaj kolhapur kolhapur mh,mh,लक्ष्मी डेवलपर्स प्राइवेट लिमिटेड
2,S2-938256871,daynamik indastrij praibhet limited,daynamik indastrij,pvt ltd,daynamikindastrij,78 a raja ram mohan roy sarani howrah calcutta wb,wb,ডায়নামিক ইন্ডাস্ট্রিজ প্রাইভেট লিমিটেড
3,S3-782716519,trading peacock health exports private limited,trading peacock health exports,pvt ltd,tradingpeacockhealthexports,no 14 fl chennai tn,tn,Trading Peacock Health Exports Private Limited
4,S3-990078959,cascade dynamic bumrungrad corporation,cascade dynamic bumrungrad,corp,cascadedynamicbumrungrad,frankfort 1060 1 2 clmnton st in,in,Cascade Dynamic Bumrungrad Corporation
5,S3-42890805,arista magnetics private limited,arista magnetics,pvt ltd,aristamagnetics,om kameswari chennai w mambalam tn,tn,Arista Magnetics Private-Limited
6,S3-897685301,kestial partners,kestial partners,,kestialpartners,8254 seneca turnpike clinton ny,ny,... Kestial Partners
7,S3-395399434,janys premier publishing,janys premier publishing,,janyspremierpublishing,twin lakes dr orange tx,tx,Jany'S Prémier Publishing


### 4.2 Blocking (`blocking.py`)

Candidate pairs are generated inside each country partition (whatever country values exist,
so France needs nothing special) as the union of:

| Bit | Pass | Why |
|---|---|---|
| 1 / 2 / 4 | exact `name_core` / `name_sorted` / `name_squash`, pool key groups ≤ 50 | cheap, exact; word swaps; domain/leet forms |
| 8 | name char 3-gram TF-IDF top-10 (cos ≥ 0.5) against pool records with ≤ 3 address tokens | typos where the address cannot help (empty / city-only addresses) |
| 16 | name + address word uni+bigram TF-IDF top-25 (cos ≥ 0.2, `max_df` 0.01) | the workhorse: renames, script names, same-name decoys ranked by their address |

Pairs are capped at 60 per S1 (exact pairs first, then by similarity). Retrieval uses
`sparse_dot_topn` (Apache-2.0) multi-threaded sparse top-k. Measured on 10k-S1 val samples
while designing it: running char 3-grams on every record costs ~3 ms per S1 (hours on test),
so that pass is restricted; word bigrams keep the address signal that `max_df` removes from
frequent unigrams (`rajendra nagar`, `5 52`). Recall is reported on the full val fold in §5.

In [4]:
pd.Series({k: v for k, v in asdict(cfg.blocking).items()})

exact_keys                               (name_core, name_sorted, name_squash)
exact_max_group                                                             50
name_char          {'column': 'name_core', 'analyzer': 'char_wb', 'ngram': ...
name_addr_word     {'column': 'name_addr', 'analyzer': 'word', 'ngram': (1,...
addr_char                                                                 None
max_per_s1                                                                  60
s1_chunk                                                                 50000
n_threads                                                                   12
vocab_sample                                                            500000
seed                                                                        42
dtype: object

### 4.3 Pair features (`features.py`)

Similarity features per candidate pair, grouped as in 07 §2. Names rank, addresses decide:
many address-agreement features (token set / Jaccard / containment, house-number and number
set agreement, region, last tokens) sit next to fuzzy name scores (rapidfuzz, MIT) on the
normalised and core names, legal-form agreement, blocking similarities and the pair's
context inside its S1 group (rank and gap to the best candidate). No country feature (open
set). Every similarity is in [0, 1]; NaN where a field is empty (LightGBM handles it).

In [5]:
pd.DataFrame([(g, ", ".join(FEATURE_COLUMNS[g])) for g in cfg.feature_groups],
             columns=["group", "features"]).assign(n=lambda d: d.features.str.count(",") + 1)

,group,features,n
0,blocking,"pass_exact, pass_name_char, pass_name_addr, sim_name_cha...",6
1,name_fuzzy,"nm_ratio, nm_partial, nm_token_sort, nm_token_set, nm_jw...",10
2,name_tokens,"tok_jaccard, tok_dice, tok_common, tok_len_l, tok_len_r,...",8
3,legal,"legal_eq, legal_missing_l, legal_missing_r",3
4,numeric,"num_jaccard, num_shared_any, num_first_eq, postcode_eq",4
5,address,"ad_token_set, ad_partial, ad_ratio, ad_jaccard, ad_conta...",8
6,context,"ctx_rank_name, ctx_gap_name, ctx_rank_addr, ctx_gap_addr...",5
7,meta,"is_s3, non_latin_r, len_ratio_name",3


### 4.4 Matching model and 4.5 decision rule (`model.py`, `decision.py`, `pipeline.fit`)

* **Training pairs**: 200k S1 entities sampled from the fit side of the inner split, each with
  all its candidates against the whole fit pool (so decoy density is realistic); label 1 for
  true pairs. Truth pairs outside the candidates cannot be learned; they are counted by the
  metric.
* **LightGBM** (MIT): binary log-loss, 63 leaves, learning rate 0.05, `min_data_in_leaf` 200,
  feature/bagging fraction 0.8, early stopping (100 rounds) on a 50k-S1 sample of the tune side;
  deterministic, 12 threads. No class reweighting: precision is bought in the decision
  layer, never by distorting probabilities.
* **Decision rule**: per S1 entity, pool-side 1-to-1 first (a pool record is kept only for its
  highest-probability S1 entity: every pool record belongs to at most one S1 in the training
  data), then keep pairs with `prob ≥ tau_abs`, `prob ≥ tau_rel · p_max`, entity emptied when
  `p_max < tau_single`, at most `max_matches`. The four thresholds are grid-searched (3,906
  rules plus a refinement) for macro F0.5 on **all** tune-side S1 entities, scored like
  inference, so singletons, blocking misses and 1-to-1 competition are all accounted for.
  Ties go to the most conservative rule (the test pool holds more decoys than train).

The cell runs the whole fit: normalisation cache, token map, blocking of the fit / stop /
tune sides, features, LightGBM, scoring of the tune side and the rule grid. The val fold is
not touched.

In [6]:
fit_timings: dict[str, float] = {}
t0 = time.time()
fitted = fit(cfg, train, ARTIFACTS, fit_timings)
timings["fit_seconds"] = fit_timings.get("fit_seconds", 0.0)
timings["tune_seconds"] = fit_timings.get("tune_seconds", 0.0)
print(f"fit() total {time.time() - t0:.0f} s; stages: {fit_timings}")
print("token map:", fitted.info["token_map_size"], "tokens, e.g.",
      list(fitted.token_map.items())[:12])
print("training:", {k: v for k, v in fitted.info["fit_info"].items()})
print("rule:", fitted.rule)

fit() total 1633 s; stages: {'normalise_seconds': 5.51, 'fit_load_seconds': 10.61, 'fit_blocking_seconds': 348.41, 'stop_load_seconds': 5.28, 'stop_blocking_seconds': 108.77, 'features_seconds': 151.41, 'fit_seconds': 335.79, 'tune_load_seconds': 5.96, 'tune_blocking_seconds': 453.13, 'score_seconds': 276.94, 'tune_seconds': 14.97}
token map: 536 tokens, e.g. [('aditia', 'aditya'), ('adity', 'aditya'), ('aigro', 'agro'), ('aiksports', 'exports'), ('ailailpi', 'llp'), ('aimtrpraijij', 'enterprises'), ('ainrji', 'energy'), ('aisais', 'ss'), ('aiti', 'it'), ('akro', 'agro'), ('alkpa', 'alpha'), ('alph', 'alpha')]
training: {'rows': 6719931, 'positive_rate': 0.09974239318826339, 'best_iteration': 1998, 'tune_logloss': 0.010413675988602895, 'tune_auc': 0.9998024954225085, 'fit_seconds': 335.79}
rule: DecisionRule(tau_abs=0.47, tau_rel=0.0, tau_single=0.52, max_matches=11, one_to_one=True)


Blocking quality of the three training-time sides (recall, candidates per S1), the 20 most
important features (gain, normalised) and the best rules of the tune grid. An address
feature missing from the top 15 would be a bug (08 §8).

In [7]:
blk = pd.DataFrame({side: fitted.info[f"{side}_blocking"] for side in ("fit", "stop", "tune")}).T
display(blk[["pair_recall", "entity_recall", "ceiling_f_beta", "candidates_mean", "candidates_p95"]])
display(fitted.matcher.importance().head(20).rename("gain").to_frame())
fitted.tune_table.sort_values("f_beta", ascending=False).head(10)

,pair_recall,entity_recall,ceiling_f_beta,candidates_mean,candidates_p95
fit,0.973580,0.997445,0.990903,33.764425,42.0
stop,0.990733,0.999216,0.997052,32.803258,60.0
tune,0.990317,0.999275,0.996959,32.963458,60.0


,gain
feature,
ad_token_set,0.399397
sim_name_addr_word,0.102814
ctx_gap_addr,0.070321
core_token_set,0.061407
ctx_rank_addr,0.056247
num_jaccard,0.052957
core_jw,0.046890
ad_jaccard,0.041013
nm_token_sort,0.025735


,tau_abs,tau_rel,tau_single,max_matches,one_to_one,f_beta,n_pred,pair_precision,pair_recall,match_rate,stage
764,0.42,0.0,0.52,11,True,0.984645,1481650,0.994804,0.965340,0.942827,grid
515,0.38,0.0,0.53,11,True,0.984634,1484433,0.994315,0.966678,0.942782,grid
638,0.40,0.0,0.50,11,True,0.984633,1483099,0.994551,0.966038,0.942927,grid
635,0.40,0.0,0.45,11,True,0.984629,1483225,0.994508,0.966078,0.943203,grid
761,0.42,0.0,0.47,11,True,0.984617,1481764,0.994762,0.965373,0.943083,grid
389,0.36,0.0,0.51,11,True,0.984616,1485885,0.994033,0.967348,0.942875,grid
641,0.40,0.0,0.55,11,True,0.984610,1482971,0.994587,0.965990,0.942655,grid
890,0.44,0.0,0.54,11,True,0.984607,1480231,0.995035,0.964639,0.942723,grid
512,0.38,0.0,0.48,11,True,0.984604,1484551,0.994272,0.966713,0.943038,grid
509,0.38,0.0,0.43,11,True,0.984600,1484680,0.994228,0.966754,0.943317,grid


## 5. Evaluation on the validation fold

`run_fold` blocks, scores and decides the val fold once with the frozen rule. Primary metric:
**macro F0.5 over all val S1 entities, singletons included** (`evaluate.score_pairs`, equal to
`metrics.breakdown`). Blocking quality: candidate pair recall, entity recall and the ceiling
F0.5 a perfect matcher would reach on these candidates.

In [8]:
t0 = time.time()
metrics, val_pairs, val_scored, val_matches = run_fold(cfg, fitted, val)
for k in ("blocking_seconds", "score_seconds", "decide_seconds", "normalise_seconds"):
    timings[k] = metrics.get(k, 0.0)
print(f"run_fold {time.time() - t0:.0f} s")
pd.Series(metrics)

run_fold 736 s


f_beta                    0.984365
f_beta_singletons         0.983998
f_beta_matched            0.984386
pair_precision            0.995210
pair_recall               0.963726
entities             441333.000000
singletons            24684.000000
cand_recall               0.990570
entity_recall             0.999330
ceiling_f_beta            0.997067
cands_mean               32.970535
cands_p95                60.000000
normalise_seconds         5.460000
blocking_seconds        448.190000
score_seconds           270.240000
decide_seconds            6.790000
dtype: float64

Recall of each blocking pass on the val fold (a pair can come from several passes) and the
standard slice report (11 §7): country, source, Indic-script pool names, ambiguous core
names, singletons, number of true matches, empty addresses.

In [9]:
is_true = pair_in(val_pairs, val.pairs)          # candidate pair is a true pair
rows = []
for name, bit in PASS_BITS.items():
    in_pass = (val_pairs["pass"].to_numpy() & bit) != 0
    if in_pass.any():
        rows.append((name, in_pass.sum() / len(val.s1), (in_pass & is_true).sum() / len(val.pairs)))
display(pd.DataFrame(rows, columns=["pass", "pairs_per_s1", "recall"]))
s1n_val = load_normalised("train", (1,), cfg, val.s1[C.ENTITY_ID], fitted.token_map)
pooln_val = load_normalised("train", (2, 3), cfg, pd.concat([val.s2, val.s3])[C.ENTITY_ID],
                            fitted.token_map)
slices = slice_report(val_matches, val, s1n_val, pooln_val)
slices.to_csv(ARTIFACTS / "slices.csv", index=False)
slices

,pass,pairs_per_s1,recall
0,exact_core,7.244736,0.564582
1,exact_sorted,7.333775,0.595844
2,exact_squash,7.311683,0.594464
3,name_char,7.387163,0.045524
4,name_addr_word,21.615297,0.982539


,family,slice,entities,f_beta,pair_precision,pair_recall,n_true,n_pred,tp
0,country,India,176522,0.983474,0.994915,0.961379,611167,590566,587563
1,country,US,264811,0.984959,0.995406,0.965291,916445,888719,884636
2,country,all,441333,0.984365,0.995210,0.963726,1527612,1479285,1472199
3,source,S2,384142,0.975956,0.995357,0.966073,739443,717688,714356
4,source,S3,388390,0.973105,0.995071,0.961523,788169,761597,757843
5,non_latin,yes,54312,0.984014,0.996985,0.952858,204745,195683,195093
6,non_latin,no,387021,0.984414,0.994939,0.965408,1322867,1283602,1277106
7,non_latin,all,441333,0.984365,0.995210,0.963726,1527612,1479285,1472199
8,domain_form,yes,82591,0.986434,0.995783,0.964432,349356,338357,336930
9,domain_form,no,358742,0.983888,0.995040,0.963516,1178256,1140928,1135269


**Harder validation** (11 §6): test has 5.8 pool records per S1 against 4.7 in train, so the
same frozen rule is also scored on a val variant that drops 20 % of the S1 entities but keeps
their pool records (they become unowned decoys). A version whose harder score falls while
val rises is buying recall with false merges.

In [10]:
harder_metrics, *_ = run_fold(cfg, fitted, harder_fold(val), tag="harder")
print({k: round(v, 4) for k, v in harder_metrics.items() if k.startswith("f_beta") or k.startswith("pair")})

{'f_beta': 0.9838, 'f_beta_singletons': 0.9808, 'f_beta_matched': 0.984, 'pair_precision': 0.9943, 'pair_recall': 0.9642}


## 6. Error analysis

Samples of the four error kinds, both records side by side with the model probability:
false merges on matched entities, missed true pairs, matched entities predicted empty
(false singletons) and predictions on true singletons. The counts say where the lost F0.5
sits; the samples name the pattern for the next version.

In [11]:
counts = {}
for kind in ("false_merge", "missed", "false_singleton", "singleton_merge"):
    sample = error_samples(val_matches, val, kind, n=10, scored=val_scored)
    counts[kind] = len(error_samples(val_matches, val, kind, n=10**9))
    print(f"--- {kind}: {counts[kind]} pairs")
    display(sample)

--- false_merge: 6685 pairs


,source1_entity_id,entity_id,prob,name_l,addr_l,name_r,addr_r
0,S1-137695892,S3-697508841,0.929165,Classic Suisse LLC,"Phoenix, 15801 48th Street, AZ, Unit 1216",Classic Suisse LLC,
1,S1-166811225,S2-152577270,0.576590,"Cascio, Smith & Drew","16-17 163 Street, Whitestone, NY","LLC Cascio, Smith & Drew","16-19 163 ST, WHITESTONE CITY, NY"
2,S1-193648498,S2-676218258,0.866776,Chamunda Brothers,"Bangalore North, Karnataka, 12Th Main, 3Rd Phase, Peenya...",Chamunda Brothers Private Limited,"NO.205/B-8A, 12TH MAIN, 3RD PHASE, PEENYA INDUSTRIAL ARE..."
3,S1-481488678,S2-931033538,0.673783,Allied Interests,"Asheville, 42 Magnolia Avenue, NC",Allied Interests LLC,
4,S1-584271054,S2-215510386,0.862161,Art Finance Co,"B-1/134 (Pvt. Cabin No.-1), Second Floor Yamuna Vihar, D...",Art Finance,
5,S1-608927025,S2-802800679,0.734608,Optimal Mobility Group,"11 Wildwood Drive, Newburgh, NY",Optimal Mobility Group LLC,"14 WILDWOOD DR, NEWBURGH, NY"
6,S1-611897728,S2-798708892,0.682256,A Direct Verde Inc.,"793 Eustace Avenue, Fort Thomas Ky, KY",A Direct Inc Center,
7,S1-613305733,S3-788406412,0.723654,Scope Trust,"1-9-646/1, To 4 Vidya Nagar, Hyderabad, Telangana",Scope Trúst Infratech,"తెలంగాణ, Hyderabad, Block D-425 1-9-646/14"
8,S1-738128729,S2-959360977,0.988262,Faclara Capital Partners LLC,"8 Web Road, Georgetown, MA",Faclara Capital Partners,"29 WEB ROAD, MA, GEORGETOWN"
9,S1-858132312,S2-948897515,0.921222,Synergia & Brothers Private Limited,"Gurgaon, 6Th Floor B-Wing, Gsc Tower, Jaipur Expy., Sohn...",Synergia & Brothers Private,


--- missed: 53904 pairs


,source1_entity_id,entity_id,prob,name_l,addr_l,name_r,addr_r
0,S1-165098328,S3-923481956,0.044353,Kaur Global Digital Inc,"115 Mckinley Avenue, Sapulpa, OK",Vantageyuma Sys,"115 Mckinley Avenue, Sapulpa, OK"
1,S1-223235489,S3-693515795,0.217243,Cardiology Tri-State Care LLC,"4313 Gerald Road, Ashtabula, OH",Cardiology Tri-State,
2,S1-422860972,S3-398485332,0.423775,Osborne & Schalk,"5325 Mandarin Circle, Chattanooga, TN",Osborne & Schalk Corp | www.osborne.com,"532 Mandarin Circle, Chattanooga, Tennessee"
3,S1-453795736,S3-136583279,0.148833,Harris Twp Wildlife Direct Initiative,"607 Rosslyn Road, Harris Twp, PA",Harris Twp Wildlife Direct Initiative LP,"606 Rosslyn Rd, Harris Twp, Pennsylvania"
4,S1-479462718,S3-844680400,0.240763,Memorial Fellowship LLC,"109 Turkey Hollow Road, Campbell County, VA",The Memorial Fellowship LLC,
5,S1-585940696,S3-706624556,0.037893,Capital Alphabet,"25 Hunting Hollow Drive, Pepper Pike, OH",Calovera Labs,"25 Hunting Hollow Drive, Pepper Pike, OH"
6,S1-634305339,S2-131572980,0.166222,Universal Exports Private Limited,"C-1, G-11, Ground Floor, Krishna Apra Plaza, Sector Alph...",यूनिवर्सल एक्सपोर्ट्स प्राइवेट लिमिटेड,"C-##1, LUCKNOW HQ REGION, उत्तर प्रदेश"
7,S1-718326235,S3-740299382,NaN,HHD Beverages Private Limited,"1/25, Hazuri Bhawan, Peepal Mandi Road, Agra, Uttar Pradesh",Ariapyra,"Agra, 1/25, UP, Agra"
8,S1-892913472,S2-937795220,0.267592,Best Beverage Holdings,"6833 Walnut Avenue, Orangevale, CA",Best Beverage,
9,S1-98315586,S2-315720146,0.088194,Delta Homecare,"1225 Osteen Street, Unit 1, Vidor, TX",Umbraectoorbi,"1225 OSTEEN ST, VIDOR, TX"


--- false_singleton: 1509 pairs


,source1_entity_id,entity_id,prob,name_l,addr_l,name_r,addr_r
0,S1-15756653,S2-359256205,0.043144,Q F & B Biotherapeutics,"3418 Park Road, Spokane Valley, WA",Q F & B Biotherapeutics Ltd,"3685 PARK ROAD, SPOKANE VALLEY, WA"
1,S1-305224497,S2-544426290,NaN,Varois,"N3937 Schielke Road, Town Of Schley, WI",SOLZETA,"SCHIELKE RD, GLEASOON, WI"
2,S1-382106823,S3-193409076,0.004971,Twisted Grill LLC,"1067 Monroe Street, Chicago, IL",Twisted 6griatl LLC,
3,S1-433616795,S3-300908171,NaN,Cox Grand Paper Inc,"39 Sobro Avenue, Hempstead, NY",Cox Grand Pmep Inc,"#39 Sobro Avenue, Valley Stream, New York"
4,S1-60374765,S2-705571307,0.030512,Straight Edge Auto Body,"727 Millers Road, Des Plaines, IL",STRAIGHT EDGE AUTO BODY INC.,"997- MILLERS RD, DES PLAINES, IL"
5,S1-74105174,S3-683300491,0.726105,Christensen Safe Drilling,"1587 Route 1, Perry, ME",Christensen Safe-Drilling,
6,S1-756901544,S3-181843208,0.180694,Best Mountain Commodities LLC,"991 Western Drive, Chanhassen, MN",Best Mountain,
7,S1-902608716,S3-482429848,0.097498,Meyers Holding Company LLC,"112 Richmond Avenue, West Haven, CT",Meyers Company LLC Center,
8,S1-919213174,S3-411199990,0.488543,Poonam Jewelery Private Limited,"C-63, Surya Nagar, Ghaziabad, Uttar Pradesh",Poonam Jewe1ery Pridnnate Limited,"C-62, Surya Nagar, Ghaziabad, UP"
9,S1-982251971,S3-887774250,0.000506,Shiv Projects,"640, Block-O New Alipore, Kolkata, Kolkata, Howrah, West...",Shiv Projects,"40, Kolkota, পশ্চিমবঙ্গ"


--- singleton_merge: 401 pairs


,source1_entity_id,entity_id,prob,name_l,addr_l,name_r,addr_r
0,S1-101467335,S3-802180227,0.916866,Ag Road Co,"C/O- Prangya Paramita Mahanta, At- Brundaban Bihar, Madh...",Ag Road,
1,S1-181116332,S2-765632443,0.948639,Smyrna Animal Hospital Inc.,"110 Creek Court, Smyrna, TN",Smyrna Animal Hospital Inc,"TN, SMYRNA, 114 CREEK CT"
2,S1-189228639,S3-977832728,0.866009,Management Db Farmer Private Limited,"Planner House C 21/87A, Mahamandal Nagar, Lahurabir, Var...",Management Db Impex Private Limited,"Planner House C 21/100A, Varanasi, UP"
3,S1-30056058,S2-957804806,0.993173,Osprey Group,"231 Silvermine Avenue, Norwalk, CT",Osprey Group,"232 Silvermine Ave, NORWALK, CT"
4,S1-318775017,S2-175392776,0.623930,Domjur Systems Ltd,"77/5/6, Benaras Road, Domjur, Howrah, West Bengal",Domjur Systems Infratech Limited - 3987383648,"77/5/27, BENARAS ROAD, DOMJUR, West Bengal"
5,S1-460620358,S3-201640348,0.827509,Metropolitan Brc,"2013 Roosevelt Street, Arlington County, VA",Metropolitan Brc,
6,S1-602701852,S2-661502079,0.865504,Brightan Dogecoin LLC,"15 Eric Clauson Lane, Falmouth, MA",Brrightan Dogecoin Llc - 4697928860,"0015 ERIC CLAUSON LN, FALMOUHT CDP, MA"
7,S1-718172744,S3-460238385,0.986475,Visoft Capital LLC,"4807 Cowslip Court, Oxon Hill, MD",Visoft Capital Partners,"4807 Cowslip Ct, Maryland, Oxon Hill"
8,S1-782452787,S2-206216664,0.688210,Smith Equity Partners LLC,"1116 7th Street, Havre, MT",SMITH EQUITY PARTNERS PARTNERS,"001121 SEVENTH ST, HAVRE, MT"
9,S1-982362384,S3-883430624,0.773560,NX Neer Ltd,"13, Satyanarayan Temple Road Salkia, Howrah, West Bengal",NU Néer Ltd,"13, Satyanarayan Temple Road Salkia, Howrah, WB"


## 7. Log the result

Records `metrics.json` and this version's row in `experiments/experiments.csv`, stamped
with the git commit of `src/` (it must not end in `-dirty`). Keys follow 13 §2.2.

In [12]:
record = {
    "hypothesis": "full V1 stack reaches >= 0.97 macro F0.5 on val with cand recall >= 0.98",
    "blocking_config": asdict(cfg.blocking), "feature_groups": list(cfg.feature_groups),
    "model_params": asdict(cfg.model), "rule": asdict(fitted.rule),
    **{k: metrics[k] for k in ("f_beta", "f_beta_singletons", "f_beta_matched",
                               "pair_precision", "pair_recall")},
    **{k: metrics[k] for k in ("cand_recall", "entity_recall", "ceiling_f_beta",
                               "cands_mean", "cands_p95")},
    "harder_f_beta": harder_metrics["f_beta"],
    "tune_f_beta": float(fitted.tune_table["f_beta"].max()),
    "n_fp": counts["false_merge"] + counts["singleton_merge"], "n_fn": counts["missed"],
    "n_false_singleton": counts["false_singleton"], "errors": counts,
    "token_map_size": fitted.info["token_map_size"],
    "best_iteration": fitted.matcher.best_iteration_,
    **timings, "peak_rss_gb": peak_rss_gb(), "decision": "KEEP",
}
row = log_result(
    EXP_DIR, change="base model: normalise+learned map, multi-pass blocking, 47 feats, "
                     "LightGBM, tuned 1-to-1 rule",
    group="INT", local_f05=metrics["f_beta"], cand_recall=metrics["cand_recall"],
    notes=(f"cands {metrics['cands_mean']:.1f}/S1; singleton F0.5 "
           f"{metrics['f_beta_singletons']:.4f}; harder {harder_metrics['f_beta']:.4f}"),
    metrics=record, owner="M1", parent="", decision="KEEP")
row

{'version': 'v001',
 'date': '2026-09-25',
 'group': 'INT',
 'change': 'base model: normalise+learned map, multi-pass blocking, 47 feats, LightGBM, tuned 1-to-1 rule',
 'local_f05': '0.9844',
 'cand_recall': '0.9906',
 'public_f05': '',
 'commit': '875079a',
 'notes': 'cands 33.0/S1; singleton F0.5 0.9840; harder 0.9838',
 'owner': 'M1',
 'parent': '',
 'decision': 'KEEP'}

## 8. Conclusion

Written after the run from the numbers above (see the markdown cell at the end of §9).

## 9. Test inference (shortlisted: upload #1)

Same pipeline on the test split: normalise (cached), block per country (France included),
score, decide with the frozen rule, write both files with `submission.write_pairs` from the
exact pairs frame that was scored. Then the sanity checks of 11 §10: one row per test S1 in
both files, every country present with candidates and matches, match rates and candidates per
S1 close to val.

In [13]:
t0 = time.time()
match_path, cand_path, s1n_test, test_matches, test_summary = run_test(cfg, fitted)
print(f"run_test {time.time() - t0:.0f} s -> {match_path}, {cand_path}")


def per_country(s1n, matches, n_cands_by_s1):
    """Match rate, matches and candidates per S1, by country (11 §10 sanity table)."""
    country = s1n.set_index(C.ENTITY_ID)[C.COUNTRY]
    n_s1 = s1n.groupby(C.COUNTRY).size()
    by = matches[C.S1_ID].map(country)
    return pd.DataFrame({
        "s1": n_s1,
        "cands_per_s1": n_cands_by_s1.groupby(n_cands_by_s1.index.map(country)).sum() / n_s1,
        "matched_share": matches.groupby(by)[C.S1_ID].nunique() / n_s1,
        "matches_per_s1": matches.groupby(by).size() / n_s1,
    })


test_table = per_country(s1n_test, test_matches, test_summary["n_cands"])
val_table = per_country(s1n_val, val_matches, val_pairs.groupby(C.S1_ID).size())
display(pd.concat({"test": test_table, "val": val_table}))
country_of = s1n_test.set_index(C.ENTITY_ID)[C.COUNTRY]
pd.crosstab(test_summary.index.map(country_of),
            pd.cut(test_summary["p_max"], [0, .1, .3, .5, .7, .9, 1.0]), normalize="index").round(3)

run_test 3053 s -> /home/suryaguru/StudioProjects/aws/business_entity_resolution/output/matching_results.tsv, /home/suryaguru/StudioProjects/aws/business_entity_resolution/output/candidate_pairs.tsv


s1  cands_per_s1  matched_share  matches_per_s1
test France  259452     37.044401       0.949925        3.398764
     India   809986     34.960182       0.940861        3.282284
     US      663106     34.072592       0.942652        3.342776
val  India   176522     33.666064       0.942098        3.345566
     US      264811     32.506897       0.942355        3.356050

p_max,"(0.0, 0.1]","(0.1, 0.3]","(0.3, 0.5]","(0.5, 0.7]","(0.7, 0.9]","(0.9, 1.0]"
row_0,,,,,,
France,0.016,0.016,0.008,0.006,0.009,0.945
India,0.039,0.012,0.005,0.004,0.005,0.934
US,0.036,0.015,0.005,0.003,0.004,0.937


Both validators on the exact files that will be uploaded: ours (`submission.validate` with
id existence checks) and the organisers' stdlib-only `validate_submission.py`.

In [14]:
out = subprocess.run([sys.executable, "-m", "entity_resolution.submission", "--output-dir",
                      str(C.OUTPUT), "--check-ids"], capture_output=True, text=True)
print(out.stdout[-2000:], out.stderr[-2000:])
out = subprocess.run([sys.executable, str(C.OFFICIAL_VALIDATOR), "--matching", str(match_path),
                      "--candidate", str(cand_path), "--test-dir", str(C.DATASET / "test")],
                     capture_output=True, text=True)
print(out.stdout[-3000:], out.stderr[-2000:])
print(f"notebook total {time.time() - t_start:.0f} s, peak RSS {peak_rss_gb()} GB")

PASS
 


ML Challenge 2026 — submission validator
  test dir: /home/suryaguru/StudioProjects/aws/business_entity_resolution/dataset/student_resource/dataset/test
  required S1 entities: 1732544
  matching_results.tsv: 1732544 rows (98922 empty, 1633622 non-empty).
  candidate_pairs.tsv: 1732544 rows (0 empty, 1732544 non-empty).

PASS — no blocking issues found. Safe to submit.
 
notebook total 6186 s, peak RSS 6.61 GB
